In [0]:
%run /Users/rahulpatel@cyntexa.com/IAP_Practice/freshkart/00_set_up

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

bronze_stores = spark.table(f"{catalog}.bronze.stores")

silver_stores = bronze_stores.select(
    upper(trim(col("store_id"))).alias("store_id"),       # string, RSU
    trim(col("store_name")).alias("store_name"),           # string, RS
    trim(col("city")).alias("city"),                       # string, RS
    trim(col("state")).alias("state"),                     # string, RS
    trim(col("region")).alias("region"),                   # string, RS
    to_date(col("opening_date")).alias("opening_date"),    # date
)

(
    silver_stores.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.stores")
)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with Change Data Feed enabled ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.products (
        product_id STRING,
        product_name STRING,
        brand STRING,
        category_l1 STRING,
        category_l2 STRING,
        packsize INT,
        pack_unit STRING,
        mrp DECIMAL(10,2),
        selling_price DECIMAL(10,2),
        is_active BOOLEAN,
        tags ARRAY<STRING>,
        last_changed_snapshot_date DATE
    )
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with SCD Type 2 columns ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.customers (
        customer_id STRING,
        full_name STRING,
        email STRING,
        phone STRING,
        city STRING,
        tier STRING,
        signup_date DATE,
        effective_from DATE,
        effective_to DATE,
        is_current BOOLEAN
    )
""")

# --- 2. Read & transform bronze customers ---
bronze_customers = spark.table(f"{catalog}.bronze.customers")

silver_customers = bronze_customers.select(
    upper(trim(col("customer_id"))).alias("customer_id"),                              # string, RSU
    trim(col("full_name")).alias("full_name"),                                   # string, RS
    lower(trim(col("email"))).alias("email"),                                    # string, RS + lower
    substring(regexp_replace(col("phone"), "[^0-9]", ""), -10, 10).alias("phone"),  # last 10 digits
    trim(col("city")).alias("city"),                                             # string, RS
    upper(trim(col("tier"))).alias("tier"),                                      # string, RSU
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    ).alias("signup_date"),                                                      # date, both formats
    to_date(regexp_extract(col("_source_file"), r"(\d{4}-\d{2}-\d{2})", 1)).alias("snapshot_date"),
)

# --- 3. Dedup by customer_id, keep latest snapshot ---
window_spec = Window.partitionBy("customer_id").orderBy(col("snapshot_date").desc())
silver_customers_dedup = (
    silver_customers
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. Determine effective_from ---
# New customers (not in silver): effective_from = signup_date (first date this version applies)
# Existing customers with tracked changes: effective_from = snapshot_date (date new version applies)
existing_ids = spark.sql(
    f"SELECT DISTINCT customer_id, True AS _exists FROM {catalog}.silver.customers WHERE is_current = true"
)

source_flagged = silver_customers_dedup.join(existing_ids, "customer_id", "left")

source_final = source_flagged.withColumn(
    "effective_from",
    when(col("_exists").isNull(), col("signup_date"))
    .otherwise(col("snapshot_date"))
).drop("_exists")

# --- 5. Identify changed customers (tracked columns only: full_name, email, phone) ---
current = spark.sql(
    f"SELECT customer_id, full_name, email, phone FROM {catalog}.silver.customers WHERE is_current = true"
)

changed_ids = (
    source_final.alias("s")
    .join(current.alias("t"), col("s.customer_id") == col("t.customer_id"), "inner")
    .filter(
        "s.full_name IS DISTINCT FROM t.full_name "
        "OR s.email IS DISTINCT FROM t.email "
        "OR s.phone IS DISTINCT FROM t.phone"
    )
    .select(col("s.customer_id").alias("customer_id"))
    .distinct()
)

# New version rows for changed customers (will be inserted after MERGE closes old rows)
new_versions = (
    source_final.join(changed_ids, "customer_id", "inner")
    .select(
        "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
        "effective_from",
        lit(None).cast("date").alias("effective_to"),
        lit(True).alias("is_current"),
    )
)

# --- 6. MERGE: close old rows (tracked changes) + update non-tracked in place + insert new customers ---
source_for_merge = source_final.select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
    "effective_from",
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current"),
)

delta_table = DeltaTable.forName(spark, f"{catalog}.silver.customers")

(
    delta_table.alias("t")
    .merge(source_for_merge.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(
        condition=(
            "t.full_name IS DISTINCT FROM s.full_name "
            "OR t.email IS DISTINCT FROM s.email "
            "OR t.phone IS DISTINCT FROM s.phone"
        ),
        set={"effective_to": "s.effective_from", "is_current": "false"}
    )
    .whenMatchedUpdate(
        condition="t.city IS DISTINCT FROM s.city OR t.tier IS DISTINCT FROM s.tier",
        set={"city": "s.city", "tier": "s.tier"}
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# --- 7. INSERT new version rows for changed customers ---
n_new_versions = new_versions.count()
if n_new_versions > 0:
    new_versions.write.mode("append").saveAsTable(f"{catalog}.silver.customers")

# print(f"New version rows inserted: {n_new_versions}")

# # --- 8. Display results ---
# display(spark.sql(f"SELECT * FROM {catalog}.silver.customers ORDER BY customer_id, effective_from"))
# --- 2. Read & transform bronze products ---
bronze_products = spark.table(f"{catalog}.bronze.products")

silver_products = bronze_products.select(
    upper(trim(col("product_id"))).alias("product_id"),                      # string, RSU
    trim(col("product_name")).alias("product_name"),                       # string
    trim(col("brand")).alias("brand"),                                      # string
    trim(get_json_object(col("category"), "$.l1")).alias("category_l1"),    # string
    trim(get_json_object(col("category"), "$.l2")).alias("category_l2"),    # string
    get_json_object(col("pack"), "$.size").cast("int").alias("packsize"),  # int
    trim(get_json_object(col("pack"), "$.unit")).alias("pack_unit"),        # string
    col("mrp").cast("decimal(10,2)").alias("mrp"),                         # decimal
    col("selling_price").cast("decimal(10,2)").alias("selling_price"),     # decimal
    col("is_active").cast("boolean").alias("is_active"),                   # boolean
    from_json(col("tags"), "array<string>").alias("tags"),                 # array<string>
    to_date(col("snapshot_date")).alias("last_changed_snapshot_date"),      # date
)

# --- 3. Dedup by product_id, keep latest snapshot ---
window_spec = Window.partitionBy("product_id").orderBy(col("last_changed_snapshot_date").desc())
silver_products_dedup = (
    silver_products
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. MERGE INTO silver (SCD Type 1) ---
# Update only when at least one value changed (excluding last_changed_snapshot_date)
delta_table = DeltaTable.forName(spark, f"{catalog}.silver.products")

update_cols = [c for c in silver_products_dedup.columns if c != "product_id"]
update_set = {c: f"s.{c}" for c in update_cols}

merge_condition = " OR ".join([
    f"t.{c} IS DISTINCT FROM s.{c}"
    for c in update_cols
    if c != "last_changed_snapshot_date"
])

(
    delta_table.alias("t")
    .merge(silver_products_dedup.alias("s"), "t.product_id = s.product_id")
    .whenMatchedUpdate(condition=merge_condition, set=update_set)
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with SCD Type 2 columns ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.customers (
        customer_id STRING,
        full_name STRING,
        email STRING,
        phone STRING,
        city STRING,
        tier STRING,
        signup_date DATE,
        effective_from DATE,
        effective_to DATE,
        is_current BOOLEAN
    )
""")

# --- 2. Read & transform bronze customers ---
bronze_customers = spark.table(f"{catalog}.bronze.customers")

silver_customers = bronze_customers.select(
    upper(trim(col("customer_id"))).alias("customer_id"),                              # string, RSU
    trim(col("full_name")).alias("full_name"),                                   # string, RS
    lower(trim(col("email"))).alias("email"),                                    # string, RS + lower
    substring(regexp_replace(col("phone"), "[^0-9]", ""), -10, 10).alias("phone"),  # last 10 digits
    trim(col("city")).alias("city"),                                             # string, RS
    upper(trim(col("tier"))).alias("tier"),                                      # string, RSU
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    ).alias("signup_date"),                                                      # date, both formats
    to_date(regexp_extract(col("_source_file"), r"(\d{4}-\d{2}-\d{2})", 1)).alias("snapshot_date"),
)

# --- 3. Dedup by customer_id, keep latest snapshot ---
window_spec = Window.partitionBy("customer_id").orderBy(col("snapshot_date").desc())
silver_customers_dedup = (
    silver_customers
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. Determine effective_from ---
# New customers (not in silver): effective_from = signup_date (first date this version applies)
# Existing customers with tracked changes: effective_from = snapshot_date (date new version applies)
existing_ids = spark.sql(
    f"SELECT DISTINCT customer_id, True AS _exists FROM {catalog}.silver.customers WHERE is_current = true"
)

source_flagged = silver_customers_dedup.join(existing_ids, "customer_id", "left")

source_final = source_flagged.withColumn(
    "effective_from",
    when(col("_exists").isNull(), col("signup_date"))
    .otherwise(col("snapshot_date"))
).drop("_exists")

# --- 5. Identify changed customers (tracked columns only: full_name, email, phone) ---
current = spark.sql(
    f"SELECT customer_id, full_name, email, phone FROM {catalog}.silver.customers WHERE is_current = true"
)

changed_ids = (
    source_final.alias("s")
    .join(current.alias("t"), col("s.customer_id") == col("t.customer_id"), "inner")
    .filter(
        "s.full_name IS DISTINCT FROM t.full_name "
        "OR s.email IS DISTINCT FROM t.email "
        "OR s.phone IS DISTINCT FROM t.phone"
    )
    .select(col("s.customer_id").alias("customer_id"))
    .distinct()
)

# New version rows for changed customers (will be inserted after MERGE closes old rows)
new_versions = (
    source_final.join(changed_ids, "customer_id", "inner")
    .select(
        "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
        "effective_from",
        lit(None).cast("date").alias("effective_to"),
        lit(True).alias("is_current"),
    )
)

# --- 6. MERGE: close old rows (tracked changes) + update non-tracked in place + insert new customers ---
source_for_merge = source_final.select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
    "effective_from",
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current"),
)

delta_table = DeltaTable.forName(spark, f"{catalog}.silver.customers")

(
    delta_table.alias("t")
    .merge(source_for_merge.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(
        condition=(
            "t.full_name IS DISTINCT FROM s.full_name "
            "OR t.email IS DISTINCT FROM s.email "
            "OR t.phone IS DISTINCT FROM s.phone"
        ),
        set={"effective_to": "s.effective_from", "is_current": "false"}
    )
    .whenMatchedUpdate(
        condition="t.city IS DISTINCT FROM s.city OR t.tier IS DISTINCT FROM s.tier",
        set={"city": "s.city", "tier": "s.tier"}
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# --- 7. INSERT new version rows for changed customers ---
n_new_versions = new_versions.count()
if n_new_versions > 0:
    new_versions.write.mode("append").saveAsTable(f"{catalog}.silver.customers")

# print(f"New version rows inserted: {n_new_versions}")

# # --- 8. Display results ---
# display(spark.sql(f"SELECT * FROM {catalog}.silver.customers ORDER BY customer_id, effective_from"))